# Lab 1 · Notebook 1 — Hugging Face pipelines ⏱ ~25 min

**Where this fits.** In class we saw that a language model is a *tokenizer → network → post-processing* chain.
Today you will use that chain through the Hugging Face `transformers` library, open it up, and finally wrap it in a small web app.

**In this notebook you will**
1. Explore the Hugging Face **Hub** (models, model cards, licences) from code.
2. Run four ready-made BERT-family models with a single line: `pipeline(...)`.
3. Deliberately **break** a model, to build intuition about what it can and cannot do.
4. Treat model outputs as ordinary data (DataFrames, timing).

> ✅ cells with `assert` are **self-checks**: if they print a green check, your answer is right.
> Cells marked **TODO** are yours to fill in. Ask a neighbour first, then the instructor.

In [1]:
# On Google Colab: uncomment, run once, then Runtime > Restart session.
# !pip install -q transformers gradio torch pandas scikit-learn huggingface_hub

import time, warnings
import torch, transformers, pandas as pd
from transformers import pipeline

warnings.filterwarnings("ignore")
DEVICE = 0 if torch.cuda.is_available() else -1   # GPU 0 if you have one, otherwise CPU (-1)
print(f"transformers {transformers.__version__} | torch {torch.__version__} | GPU: {torch.cuda.is_available()}")

# The four small models we use all lab. Each is a distilled / BERT-style encoder (~250 MB).
SENT_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"   # sentiment (positive/negative)
NER_MODEL  = "dslim/distilbert-NER"                                         # named entities (people, places, orgs)
FILL_MODEL = "distilbert/distilbert-base-uncased"                           # masked language model (fill the blank)
ZS_MODEL   = "typeform/distilbert-base-uncased-mnli"                        # zero-shot classification (via entailment)

/private/tmp/claude-501/-Users-anasr-Desktop-addingknowledgetollms-v2/bd9011ba-489e-4b8b-ace2-cdebb5581885/scratchpad/labtest/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1022 21:49:19.291000 54479 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


transformers 5.20.2 | torch 2.14.1 | GPU: False


## 1 · The Hub, from code
The Hub is “GitHub for models”: every model has an id (`owner/name`), a **model card** (what it is, how it was trained),
a **task** (`pipeline_tag`) and — importantly — a **licence**. Remember the licence slide: not every model may be used for every purpose.

In [2]:
from huggingface_hub import HfApi
api = HfApi()

# The 5 most-downloaded text-classification models right now
for m in api.list_models(pipeline_tag="text-classification", sort="downloads", limit=5):
    print(f"{m.id:55s} downloads={m.downloads:,}")

BAAI/bge-reranker-v2-m3                                 downloads=15,068,393
SamLowe/roberta-base-go_emotions                        downloads=4,652,348
ProsusAI/finbert                                        downloads=4,020,358
BAAI/bge-reranker-base                                  downloads=3,147,188
distilbert/distilbert-base-uncased-finetuned-sst-2-english downloads=2,632,606


In [3]:
# TODO 1 — write describe_model(model_id) that returns a dict: id, task, license, downloads
def describe_model(model_id):
    info = api.model_info(model_id)
    card = info.card_data
    return {
        "id": info.id,
        "task": info.pipeline_tag,
        "license": getattr(card, "license", None) if card else None,
        "downloads": info.downloads,
    }

In [4]:
d = describe_model(SENT_MODEL)
assert d["task"] == "text-classification" and d["license"] == "apache-2.0", d
print("✅ TODO 1 passed\n")
for mid in [SENT_MODEL, NER_MODEL, FILL_MODEL, ZS_MODEL]:
    print(describe_model(mid))

✅ TODO 1 passed



{'id': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english', 'task': 'text-classification', 'license': 'apache-2.0', 'downloads': 2632606}


{'id': 'dslim/distilbert-NER', 'task': 'token-classification', 'license': 'apache-2.0', 'downloads': 143601}


{'id': 'distilbert/distilbert-base-uncased', 'task': 'fill-mask', 'license': 'apache-2.0', 'downloads': 6660845}


{'id': 'typeform/distilbert-base-uncased-mnli', 'task': 'zero-shot-classification', 'license': None, 'downloads': 237251}


**Discuss (1 min).** One of the four models printed `license: None`. Would you ship it in a commercial product without checking? What would you do?

## 2 · Your first pipeline: sentiment analysis
`pipeline(task, model=...)` downloads the model once (cached in `~/.cache/huggingface`) and returns a callable.

In [5]:
clf = pipeline("sentiment-analysis", model=SENT_MODEL, device=DEVICE)

reviews = [
    "I absolutely loved this film, the acting was brilliant.",
    "The service was slow and the food was cold.",
    "It was okay, nothing special.",
]
for r, out in zip(reviews, clf(reviews)):
    print(f"{out['label']:9s} {out['score']:.3f}  <- {r}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8556.95it/s]

POSITIVE  1.000  <- I absolutely loved this film, the acting was brilliant.
NEGATIVE  1.000  <- The service was slow and the food was cold.
NEGATIVE  0.982  <- It was okay, nothing special.


Notice the third review: the model has **only two labels**, so it must pick one even when the honest answer is *neutral*.
`top_k=None` returns the score of *every* label, not just the winner:

In [6]:
clf("It was okay, nothing special.", top_k=None)

[{'label': 'NEGATIVE', 'score': 0.9820851683616638},
 {'label': 'POSITIVE', 'score': 0.017914844676852226}]

### 🔨 TODO 2 — break the model (5 min)
A good engineer knows where a model fails. Write **one input per category** that you believe will *fool* the model,
and put the label **you** think is correct. Categories: negation, sarcasm, mixed feelings, another language, and domain jargon.
The cell after it runs the model and shows where it is wrong or unsure.

In [7]:
tricky = {
    "negation":       ("The film was not bad at all, I did not regret watching it.", "POSITIVE"),
    "sarcasm":        ("Oh great, another software update that breaks everything. Just what I needed.", "NEGATIVE"),
    "mixed":          ("Gorgeous screen and fast, but the battery is awful and support was useless.", "NEGATIVE"),
    "other language": ("Un service horrible, je ne reviendrai jamais.", "NEGATIVE"),
    "domain jargon":  ("The tumour shrank significantly after the first cycle of treatment.", "POSITIVE"),
}

In [8]:
assert all(text.strip() for text, _ in tricky.values()), "Fill in every text first"
rows = []
for category, (text, gold) in tricky.items():
    out = clf(text)[0]
    rows.append({"category": category, "text": text[:60] + ("…" if len(text) > 60 else ""),
                 "gold": gold, "predicted": out["label"], "confidence": round(out["score"], 3),
                 "correct": out["label"] == gold})
df = pd.DataFrame(rows)
display(df)
print(f"Fooled the model {int((~df.correct).sum())} / {len(df)} times")

,category,text,gold,predicted,confidence,correct
0,negation,"The film was not bad at all, I did not regret ...",POSITIVE,POSITIVE,0.995,True
1,sarcasm,"Oh great, another software update that breaks ...",NEGATIVE,POSITIVE,0.994,False
2,mixed,"Gorgeous screen and fast, but the battery is a...",NEGATIVE,NEGATIVE,0.999,True
3,other language,"Un service horrible, je ne reviendrai jamais.",NEGATIVE,NEGATIVE,0.996,True
4,domain jargon,The tumour shrank significantly after the firs...,POSITIVE,NEGATIVE,0.999,False


Fooled the model 2 / 5 times


**Write 2 sentences in this cell:** which category was hardest, and *why* do you think so (think: training data, tokenization, only two labels)?

_Your answer:_

## 3 · Three more tasks, same one-liner
The *task string* tells `pipeline` which head and post-processing to use. The encoder body is BERT-family in all of them.

In [9]:
# (a) Named-entity recognition: label each word span as person / organisation / location
ner = pipeline("token-classification", model=NER_MODEL, aggregation_strategy="simple", device=DEVICE)
text = "Angela Merkel visited the Google office in Zurich last Tuesday."
for e in ner(text):
    print(f"{e['entity_group']:4s} {e['word']:15s} score={float(e['score']):.3f}  chars {e['start']}-{e['end']}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 7763.14it/s]

PER  Angela Merkel   score=0.999  chars 0-13
ORG  Google          score=0.992  chars 26-32
LOC  Zurich          score=0.998  chars 43-49


In [10]:
# (b) Masked language modelling: the pre-training task of BERT itself
fill = pipeline("fill-mask", model=FILL_MODEL, device=DEVICE)
print("mask token for this model:", fill.tokenizer.mask_token)       # not every model uses [MASK]!
for r in fill("Paris is the [MASK] of France.", top_k=5):
    print(f"{r['token_str']:12s} {r['score']:.3f}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7301.82it/s]

mask token for this model: [MASK]


capital      0.982
birthplace   0.003
northernmost 0.001
centre       0.001
southernmost 0.001


In [11]:
# (c) Zero-shot classification: choose among labels the model was never trained on.
#     Trick: each label becomes a hypothesis ("This example is billing.") and the model checks entailment.
zs = pipeline("zero-shot-classification", model=ZS_MODEL, device=DEVICE)
out = zs("My card was charged twice this month", candidate_labels=["billing", "technical issue", "shipping"])
print(list(zip(out["labels"], [round(s, 3) for s in out["scores"]])))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7487.90it/s]

[('billing', 0.43), ('shipping', 0.292), ('technical issue', 0.278)]


### 🔨 TODO 3 — zero-shot is *prompt engineering* (6 min)
Zero-shot results depend on the **hypothesis template**. Build a mini support-ticket classifier, then compare two templates.
Write 4 tickets (one per label) and finish `classify_ticket`.

In [12]:
labels = ["billing", "technical issue", "shipping", "account"]
tickets = [
    ("I was charged twice for my subscription this month.", "billing"),
    ("The app crashes whenever I tap the upload button.", "technical issue"),
    ("My parcel has not arrived and the tracking is stuck.", "shipping"),
    ("How do I change the email address linked to my login?", "account"),
]

def classify_ticket(text, labels, template="This example is {}."):
    out = zs(text, candidate_labels=labels, hypothesis_template=template)
    return out["labels"][0], round(out["scores"][0], 3)

In [13]:
assert all(t.strip() for t, _ in tickets), "Write all 4 tickets first"
label, score = classify_ticket(tickets[0][0], labels)
assert label in labels and 0 <= score <= 1
print("✅ TODO 3 passed\n")

for template in ["This example is {}.", "This customer message is about {}."]:
    hits = sum(classify_ticket(t, labels, template)[0] == gold for t, gold in tickets)
    print(f"{hits}/{len(tickets)} correct with template: {template!r}")

✅ TODO 3 passed



3/4 correct with template: 'This example is {}.'


1/4 correct with template: 'This customer message is about {}.'


_Did the template matter? Try a third template of your own in a new cell._

## 4 · Outputs are just data — and speed matters
Pipelines return plain Python lists/dicts, so you can drop them straight into pandas.
Passing a **list** lets the pipeline batch the inputs, which is much faster than calling it in a loop.

In [14]:
texts = ["This is a really great product, I use it every day."] * 64

t0 = time.time(); _ = [clf(t) for t in texts]; loop_s = time.time() - t0
t0 = time.time(); _ = clf(texts, batch_size=16);  batch_s = time.time() - t0
print(f"loop: {loop_s:.2f}s   batched: {batch_s:.2f}s   speed-up x{loop_s / batch_s:.1f}")

pd.DataFrame(clf(reviews)).assign(text=reviews)

loop: 1.92s   batched: 0.23s   speed-up x8.5


,label,score,text
0,POSITIVE,0.999884,"I absolutely loved this film, the acting was b..."
1,NEGATIVE,0.999742,The service was slow and the food was cold.
2,NEGATIVE,0.982085,"It was okay, nothing special."


## ✅ Checkpoint — you should now be able to say
* what a *pipeline* is and how to pick a model for a task from the Hub;
* where to look for a model's **licence** and why it matters;
* three situations where a small sentiment model fails;
* why zero-shot classification is sensitive to *wording*.

**Next:** Notebook 2 — we open the black box (tokens, tensors, logits).